In [1]:
!python -m pip install reverse_geocoder


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 1: Install & Import
%pip install pandas numpy requests folium global-land-mask
import os

# Create project folders
data_directories = ['data/raw', 'data/processed']
for directory in data_directories:
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory created: {directory}")
    else:
        print(f"Directory verified: {directory}")

print("✅ Environment Ready.")

Note: you may need to restart the kernel to use updated packages.
Directory verified: data/raw
Directory verified: data/processed
✅ Environment Ready.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Cell 2: Data Acquisition & Strict India Border Validation
import pandas as pd
import numpy as np
import requests
import time
import reverse_geocoder as rg 
from io import StringIO
from datetime import datetime, timedelta, timezone
from global_land_mask import globe
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- CONFIGURATION ---
NASA_URL = "https://firms.modaps.eosdis.nasa.gov/data/active_fire/suomi-npp-viirs-c2/csv/SUOMI_VIIRS_C2_South_Asia_24h.csv"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
BATCH_SIZE = 100
MAX_RETRIES = 3

# Setup Session
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

# 1. FETCH & FILTER NASA DATA (Strict India Only)
print("📡 Contacting NASA...")
try:
    response = session.get(NASA_URL, timeout=30)
    df = pd.read_csv(StringIO(response.text))
    
    # A. Rough Box Filter (Fast)
    df = df[(df['latitude'] >= 6) & (df['latitude'] <= 38) & 
            (df['longitude'] >= 68) & (df['longitude'] <= 98)].copy()
    
    # B. Strict Border Check (Accurate)
    print(f"🧐 Verifying {len(df)} fire points against exact Indian borders...")
    if not df.empty:
        coords = list(zip(df['latitude'], df['longitude']))
        results = rg.search(coords) # Returns list of dicts with country codes
        
        # Keep only points where cc (Country Code) is 'IN'
        df['country'] = [x['cc'] for x in results]
        df_india = df[df['country'] == 'IN'].copy()
        
        # --- FIX: ADD THE LABEL HERE ---
        df_india['fire_detected'] = 1 
        # -------------------------------
        
        print(f"✅ Fires kept: {len(df_india)} (Removed {len(df) - len(df_india)} foreign points)")
    else:
        df_india = pd.DataFrame()

except Exception as e:
    print(f"❌ Error: {e}")
    df_india = pd.DataFrame()

# 2. GENERATE SAFE POINTS (Strictly India)
if not df_india.empty:
    target_count = len(df_india)
    print(f"⚖️ Generating {target_count} Safe Points (Strictly Inside India)...")
    
    safe_points = []
    
    # Loop until we have enough Valid Indian Points
    while len(safe_points) < target_count:
        # Generate a large batch (3x needed) because many will fall in ocean/neighbors
        needed = target_count - len(safe_points)
        batch_size = max(needed * 3, 50) 
        
        lats = np.random.uniform(6.0, 38.0, batch_size)
        lons = np.random.uniform(68.0, 98.0, batch_size)
        
        # Filter 1: Land Check (Fast)
        is_land = globe.is_land(lats, lons)
        land_lats = lats[is_land]
        land_lons = lons[is_land]
        
        if len(land_lats) > 0:
            # Filter 2: Border Check (Slower, precise)
            check_coords = list(zip(land_lats, land_lons))
            geo_results = rg.search(check_coords)
            
            for i, res in enumerate(geo_results):
                if res['cc'] == 'IN': # ONLY INDIA
                    safe_points.append({
                        'latitude': land_lats[i],
                        'longitude': land_lons[i],
                        'acq_date': datetime.now(timezone.utc).strftime('%Y-%m-%d'),
                        'fire_detected': 0
                    })
                    if len(safe_points) >= target_count:
                        break
        
        print(f"   > Progress: {len(safe_points)}/{target_count} safe points...")

    safe_df = pd.DataFrame(safe_points)
    
    # Combine
    master_df = pd.concat([df_india[['latitude', 'longitude', 'acq_date', 'fire_detected']], safe_df], ignore_index=True)
    print(f"✅ Final Dataset: {len(master_df)} rows (100% Inside India)")
    
    # 3. WEATHER ENRICHMENT
    weather_data = []
    print(f"🚀 Starting Weather Enrichment for {len(master_df)} points...")
    
    for i in range(0, len(master_df), BATCH_SIZE):
        batch = master_df.iloc[i : i + BATCH_SIZE]
        params = {
            "latitude": ",".join(batch['latitude'].astype(str)),
            "longitude": ",".join(batch['longitude'].astype(str)),
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,soil_moisture_0_to_7cm"
        }
        
        success = False
        for attempt in range(MAX_RETRIES):
            try:
                r = session.get(WEATHER_URL, params=params, timeout=30)
                if r.status_code == 429: raise ValueError("Rate Limit")
                r.raise_for_status()
                data = r.json()
                
                # Handle single vs list response
                if isinstance(data, list):
                    weather_data.extend([x.get('current', {}) for x in data])
                else:
                    weather_data.append(data.get('current', {}))
                success = True
                time.sleep(1.5) # Polite delay
                break
            except Exception as e:
                print(f"   ⚠️ Retrying batch {i}: {e}")
                time.sleep(5)
        
        if not success:
            weather_data.extend([{} for _ in range(len(batch))]) # Fill empty on fail
            
    # Final Merge
    weather_df = pd.DataFrame(weather_data)
    final_df = pd.concat([master_df, weather_df], axis=1)
    final_df.dropna(subset=['temperature_2m'], inplace=True) # Drop failed calls
    
    final_df.to_csv('data/raw/master_dataset_real.csv', index=False)
    print("✅ DONE. Live Data Saved.")

📡 Contacting NASA...
🧐 Verifying 3901 fire points against exact Indian borders...
Loading formatted geocoded file...
✅ Fires kept: 2306 (Removed 1595 foreign points)
⚖️ Generating 2306 Safe Points (Strictly Inside India)...
   > Progress: 2054/2306 safe points...
   > Progress: 2281/2306 safe points...
   > Progress: 2306/2306 safe points...
✅ Final Dataset: 4612 rows (100% Inside India)
🚀 Starting Weather Enrichment for 4612 points...
   ⚠️ Retrying batch 600: Rate Limit
   ⚠️ Retrying batch 600: Rate Limit
   ⚠️ Retrying batch 600: Rate Limit
   ⚠️ Retrying batch 700: Rate Limit
   ⚠️ Retrying batch 700: Rate Limit
   ⚠️ Retrying batch 700: Rate Limit
   ⚠️ Retrying batch 1400: Rate Limit
   ⚠️ Retrying batch 1400: Rate Limit
   ⚠️ Retrying batch 1400: Rate Limit
   ⚠️ Retrying batch 1500: Rate Limit
   ⚠️ Retrying batch 1500: Rate Limit
   ⚠️ Retrying batch 1500: Rate Limit
   ⚠️ Retrying batch 1600: Rate Limit
   ⚠️ Retrying batch 1600: Rate Limit
   ⚠️ Retrying batch 1600: Rate Li

In [4]:
# Cell 3: Geospatial Validation
import folium
from global_land_mask import globe
import pandas as pd

try:
    df = pd.read_csv('data/raw/master_dataset_real.csv')
    
    # Filter for Land Only
    df['is_land'] = globe.is_land(df['latitude'], df['longitude'])
    df_final = df[df['is_land']].copy()
    
    print(f"Validation: {len(df_final)} points confirmed on land.")
    print(f"Fires: {len(df_final[df_final['fire_detected']==1])} | Safe: {len(df_final[df_final['fire_detected']==0])}")

    # Map
    m = folium.Map(location=[20.59, 78.96], zoom_start=5)
    
    # Plot Safe (Green)
    for _, row in df_final[df_final['fire_detected']==0].iterrows():
        folium.CircleMarker([row['latitude'], row['longitude']], radius=4, color='green', fill=True, fill_opacity=0.4).add_to(m)
        
    # Plot Fire (Red - On Top)
    for _, row in df_final[df_final['fire_detected']==1].iterrows():
        folium.CircleMarker([row['latitude'], row['longitude']], radius=6, color='red', fill=True, fill_color='red', fill_opacity=0.8, popup=f"FRP: {row.get('frp')}").add_to(m)
        
    m.save('data/raw/geospatial_validation_report.html')
    print("✅ Map saved to 'data/raw/geospatial_validation_report.html'")
    
except FileNotFoundError:
    print("❌ Run Cell 2 first!")

Validation: 3209 points confirmed on land.
Fires: 1797 | Safe: 1412
✅ Map saved to 'data/raw/geospatial_validation_report.html'
